# Análisis experimental de ResNet en la detección de enfermedades de la papa
### Profundidad, conexiones residuales, transfer learning, fine-tuning, cambio de dominio y Grad-CAM

**Contexto regional:** la papa es un cultivo clave en Apurímac (Andahuaylas, Abancay, Chincheros) y el tizón tardío es una de sus principales amenazas. Este notebook usa **solo datasets abiertos** (sin trabajo de campo).

**Estructura**

| Bloque | Dataset | Pregunta |
|---|---|---|
| A | CIFAR-10 | ¿Las conexiones residuales resuelven el problema de degradación? (planas vs. ResNet 18/34) |
| B | PlantVillage (papa) | ¿Qué rinde mejor: entrenar desde cero, *feature extraction* (transfer learning) o *fine-tuning*? ¿Y qué arquitectura (ResNet-18/34/50, ResNeXt-50, Wide ResNet-50)? |
| C | PlantVillage + dataset externo | ¿Cuánto cae el rendimiento ante un cambio de dominio? ¿Ayuda el data augmentation fuerte o mezclar datasets? |
| D | Ambos dominios | ¿La red mira la lesión o el fondo? (Grad-CAM) |
| E | Resumen | Tablas, gráficas, parámetros, FLOPs y exportación a ONNX (despliegue en celular) |

> **Antes de ejecutar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.
> Con `QUICK_MODE = True` todo corre en pocos minutos (1 semilla, pocas épocas) para probar el flujo. Para los resultados finales de la monografía usa `QUICK_MODE = False` (3 semillas).

## 0. Instalación y configuración

In [ ]:
!pip -q install grad-cam thop kagglehub seaborn onnx

In [ ]:
import os, time, copy, random, glob, json, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ===================== CONFIGURACIÓN GENERAL =====================
QUICK_MODE = True          # True: prueba rápida | False: experimento final
USE_DRIVE  = False         # True: guarda resultados en Google Drive

if QUICK_MODE:
    SEEDS, EPOCHS_A, EPOCHS_B = [0], 10, 8
else:
    SEEDS, EPOCHS_A, EPOCHS_B = [0, 1, 2], 30, 15

BATCH_A, BATCH_B, IMG = 128, 32, 224
NUM_WORKERS = 2
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

SAVE_DIR = "/content/resultados"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/monografia_resnet"
os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print("Torch", torch.__version__, "| torchvision", torchvision.__version__, "| dispositivo:", DEVICE)
if DEVICE.type == "cpu":
    print("⚠️ No hay GPU activa. Activa una GPU en el menú de Entorno de ejecución.")

## 1. Funciones comunes: entrenamiento, evaluación y métricas

In [ ]:
def set_train_mode(model):
    """Modo train, pero las capas BatchNorm totalmente congeladas se dejan en eval
    (importante en feature extraction para no alterar las estadísticas preentrenadas)."""
    model.train()
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm):
            ps = list(m.parameters())
            if ps and not any(p.requires_grad for p in ps):
                m.eval()

@torch.no_grad()
def predict(model, loader):
    model.eval()
    P, Y = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(x)
        P.append(out.argmax(1).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def fit(model, train_loader, val_loader, epochs, max_lr, param_groups=None,
        wd=5e-4, select_best=True, verbose=True, tag=""):
    """SGD + momentum con scheduler OneCycle. Devuelve (historial, modelo).
    select_best=True restaura los pesos de la mejor época según val_loader."""
    model.to(DEVICE)
    params = param_groups if param_groups is not None else [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=wd, nesterov=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, epochs=epochs,
                                                steps_per_epoch=len(train_loader), pct_start=0.2)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    hist, best_acc, best_state = [], -1, None
    for ep in range(epochs):
        set_train_mode(model); t0 = time.time()
        tl = tc = n = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x); loss = F.cross_entropy(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); n += len(y)
        p, yv = predict(model, val_loader)
        va = accuracy_score(yv, p)
        hist.append(dict(epoch=ep + 1, train_loss=tl / n, train_acc=tc / n, val_acc=va, time=time.time() - t0))
        if select_best and va > best_acc:
            best_acc = va; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"{tag} ep {ep+1:02d}/{epochs} | loss {tl/n:.3f} | train {tc/n:.3f} | val {va:.3f} | {time.time()-t0:.0f}s")
    if select_best and best_state is not None:
        model.load_state_dict(best_state)
    return hist, model

def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_flops(model, size):
    from thop import profile
    m = copy.deepcopy(model).to(DEVICE).eval()
    x = torch.randn(1, 3, size, size, device=DEVICE)
    flops, _ = profile(m, inputs=(x,), verbose=False)
    return flops * 2 / 1e9   # thop devuelve MACs; FLOPs ≈ 2*MACs (en GFLOPs)

## BLOQUE A — Degradación: redes planas vs. residuales (CIFAR-10)

Se implementa **la misma arquitectura** con y sin conexiones residuales (`residual=False/True`), en versión adaptada a CIFAR (conv 3×3 inicial, sin max-pool). Con las mismas capas y los mismos hiperparámetros, la diferencia se debe únicamente a los *skip connections*.

Resultado esperado (He et al., 2015): la red plana de 34 capas tiene **mayor error de entrenamiento** que la de 18, mientras que ResNet-34 mejora a ResNet-18.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride, residual):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);    self.bn2 = nn.BatchNorm2d(cout)
        self.residual = residual
        self.shortcut = nn.Identity()
        if residual and (stride != 1 or cin != cout):
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            out = out + self.shortcut(x)
        return F.relu(out)

class CifarNet(nn.Module):
    CFG = {18: [2, 2, 2, 2], 34: [3, 4, 6, 3]}
    def __init__(self, depth=18, residual=True, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        layers, cin = [], 64
        for i, (cout, n) in enumerate(zip([64, 128, 256, 512], self.CFG[depth])):
            for j in range(n):
                layers.append(BasicBlock(cin, cout, 2 if (j == 0 and i > 0) else 1, residual)); cin = cout
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

# Datos CIFAR-10
mean_c, std_c = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
tf_tr = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
                            transforms.ToTensor(), transforms.Normalize(mean_c, std_c)])
tf_te = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean_c, std_c)])
cif_tr = datasets.CIFAR10("/content/data", train=True,  download=True, transform=tf_tr)
cif_te = datasets.CIFAR10("/content/data", train=False, download=True, transform=tf_te)
loader_A_tr = DataLoader(cif_tr, BATCH_A, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
loader_A_te = DataLoader(cif_te, 256, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(len(cif_tr), "imágenes de entrenamiento |", len(cif_te), "de prueba")

In [ ]:
RES_A, HIST_A = [], {}
for depth in [18, 34]:
    for residual in [False, True]:
        name = f"{'ResNet' if residual else 'Plain'}-{depth}"
        for seed in SEEDS:
            set_seed(seed)
            model = CifarNet(depth, residual)
            if seed == SEEDS[0]:
                tot, _ = count_params(model); gf = count_flops(model, 32)
            t0 = time.time()
            # select_best=False: se reporta la ÚLTIMA época (sin seleccionar sobre test)
            hist, model = fit(model, loader_A_tr, loader_A_te, EPOCHS_A, max_lr=0.1,
                              select_best=False, verbose=False, tag=name)
            HIST_A[(name, seed)] = hist
            RES_A.append(dict(modelo=name, seed=seed, params_M=tot / 1e6, GFLOPs=gf,
                              train_acc=hist[-1]["train_acc"], test_acc=hist[-1]["val_acc"],
                              train_err=1 - hist[-1]["train_acc"], test_err=1 - hist[-1]["val_acc"],
                              t_epoca_s=np.mean([h["time"] for h in hist])))
            print(f"{name} seed {seed}: train_acc={hist[-1]['train_acc']:.4f} test_acc={hist[-1]['val_acc']:.4f} ({time.time()-t0:.0f}s)")
df_A = pd.DataFrame(RES_A)
df_A.to_csv(f"{SAVE_DIR}/bloque_A.csv", index=False)
tabla_A = df_A.groupby("modelo").agg(["mean", "std"]).round(4)[["params_M", "GFLOPs", "train_err", "test_err", "t_epoca_s"]]
tabla_A

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
colors = {"Plain-18": "#e9967a", "Plain-34": "#c0392b", "ResNet-18": "#85c1e9", "ResNet-34": "#1f4e79"}
for name in colors:
    H = [pd.DataFrame(HIST_A[(name, s)]) for s in SEEDS]
    tr = np.mean([1 - h.train_acc for h in H], axis=0); te = np.mean([1 - h.val_acc for h in H], axis=0)
    ax[0].plot(range(1, len(tr) + 1), tr, label=name, color=colors[name])
    ax[1].plot(range(1, len(te) + 1), te, label=name, color=colors[name])
ax[0].set_title("Error de ENTRENAMIENTO (evidencia de degradación)"); ax[1].set_title("Error de PRUEBA")
for a in ax: a.set_xlabel("Época"); a.set_ylabel("Error"); a.legend()
ax[0].set_yscale("log")
plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/fig_A_degradacion.png", dpi=200); plt.show()

## BLOQUE B — Datos: PlantVillage (papa) y dataset externo

- **Entrenamiento / validación / prueba interna:** clases de papa de **PlantVillage** (`emmarex/plantdisease` en Kaggle): *Early blight*, *Late blight*, *Healthy*.
- **Prueba externa (cambio de dominio real):** un dataset de papa con origen distinto (fotos en condiciones no controladas). Configúralo en la siguiente celda. Opciones:
  - un dataset de Kaggle (`EXTERNAL_KAGGLE_SLUG = "usuario/nombre-dataset"`), **o**
  - una carpeta propia en Colab/Drive (`EXTERNAL_PATH`), por ejemplo el dataset *Potato Leaf (Healthy and Late Blight)* de Mendeley Data (Holeta, Ethiopia; fotos en campo, 2 clases) o *PlantDoc* (clases de papa).
- Las etiquetas se **homologan por palabras clave** en el nombre de la carpeta (`early`, `late`, `health`). Si el dataset externo no trae alguna clase, las métricas se calculan solo con las clases presentes.
- Además se crea un **proxy de domain shift** (`proxy`): el test de PlantVillage con deformaciones fuertes (fondo, color, desenfoque, ruido). Sirve como referencia cuando no hay dataset externo, **pero no lo reemplaza**: declara siempre cuál usaste.

> Para descargar de Kaggle con `kagglehub` puede pedirte iniciar sesión; si falla, sube tu `kaggle.json` o descarga el zip manualmente.

In [ ]:
import kagglehub

CLASSES = ["Early_blight", "Late_blight", "Healthy"]; NC = 3
EXTERNAL_KAGGLE_SLUG = ""   # ej: "usuario/potato-leaf-dataset"  (verifica el nombre exacto en Kaggle)
EXTERNAL_PATH        = ""   # ej: "/content/drive/MyDrive/datasets/potato_externo"

def label_from_name(name):
    n = name.lower()
    if "early" in n:  return 0
    if "late" in n:   return 1
    if "health" in n: return 2
    return None

def collect_items(root, require_potato=False):
    items, seen = [], set()
    for d, _, files in os.walk(root):
        base = os.path.basename(d)
        if require_potato and "potato" not in base.lower():
            continue
        lab = label_from_name(base)
        if lab is None:
            continue
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                p = os.path.join(d, f)
                key = (lab, f, os.path.getsize(p))     # evita duplicados (carpetas anidadas)
                if key not in seen:
                    seen.add(key); items.append((p, lab))
    return items

def show_counts(items, name):
    c = np.bincount([y for _, y in items], minlength=NC)
    print(f"{name}: {len(items)} imágenes | " + ", ".join(f"{CLASSES[i]}={c[i]}" for i in range(NC)))

# ---- PlantVillage (papa)
pv_root = kagglehub.dataset_download("emmarex/plantdisease")
pv_items = collect_items(pv_root, require_potato=True)
show_counts(pv_items, "PlantVillage-papa")
assert len(pv_items) > 1000, "No se encontraron las imágenes de papa; revisa la ruta de descarga: " + str(pv_root)

# ---- Dataset externo (opcional pero MUY recomendado)
ext_items = []
if EXTERNAL_KAGGLE_SLUG:
    ext_root = kagglehub.dataset_download(EXTERNAL_KAGGLE_SLUG); ext_items = collect_items(ext_root)
elif EXTERNAL_PATH:
    ext_items = collect_items(EXTERNAL_PATH)
if ext_items: show_counts(ext_items, "Externo")
else: print("⚠️ Sin dataset externo: solo se usará el proxy de domain shift. Configura EXTERNAL_* para el análisis completo.")

In [ ]:
# Particiones FIJAS (independientes de la semilla de entrenamiento): 70/15/15 estratificado
y_pv = [y for _, y in pv_items]
pv_train, tmp = train_test_split(pv_items, test_size=0.30, stratify=y_pv, random_state=42)
pv_val, pv_test = train_test_split(tmp, test_size=0.50, stratify=[y for _, y in tmp], random_state=42)
print(f"train={len(pv_train)} val={len(pv_val)} test={len(pv_test)}")

# Externo: mitad para evaluar (siempre) y mitad para "mezclar" en el experimento C2
ext_mix, ext_test = [], []
if ext_items:
    try:
        ext_mix, ext_test = train_test_split(ext_items, test_size=0.5, stratify=[y for _, y in ext_items], random_state=42)
    except ValueError:
        ext_mix, ext_test = train_test_split(ext_items, test_size=0.5, random_state=42)
    show_counts(ext_test, "Externo-test")

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
norm = transforms.Normalize(MEAN, STD)

class ImgDS(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), y

eval_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), norm])

TF = {
  "weak": transforms.Compose([transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0)),
                              transforms.RandomHorizontalFlip(), transforms.ToTensor(), norm]),
  "strong": transforms.Compose([transforms.RandomResizedCrop(IMG, scale=(0.35, 1.0)),
                                transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                                transforms.RandomRotation(35, fill=(90, 70, 50)),
                                transforms.ColorJitter(0.5, 0.5, 0.5, 0.08),
                                transforms.RandomGrayscale(0.1),
                                transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 2.0))], p=0.4),
                                transforms.ToTensor(), norm,
                                transforms.RandomErasing(p=0.4, scale=(0.02, 0.25), value="random")]),
}

class AddNoise:
    def __init__(self, s=0.08): self.s = s
    def __call__(self, t): return t + torch.randn_like(t) * self.s

# Proxy de domain shift: test de PlantVillage con fondo/color/desenfoque/ruido alterados
proxy_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomAffine(40, translate=(0.2, 0.2), scale=(0.6, 1.2), fill=(95, 75, 55)),
    transforms.ColorJitter(0.5, 0.5, 0.5, 0.05),
    transforms.GaussianBlur(5, (0.5, 2.0)),
    transforms.ToTensor(), norm, AddNoise(0.08)])

def mk(items, tf, shuffle=False, bs=BATCH_B, drop_last=False):
    return DataLoader(ImgDS(items, tf), bs, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True, drop_last=drop_last)

LOADERS_EVAL = {"intra": mk(pv_test, eval_tf, bs=64), "proxy": mk(pv_test, proxy_tf, bs=64)}
if ext_test: LOADERS_EVAL["externo"] = mk(ext_test, eval_tf, bs=64)
val_loader = mk(pv_val, eval_tf, bs=64)
print("Evaluaciones disponibles:", list(LOADERS_EVAL))

# Vista rápida de ambos dominios
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for j in range(4):
    p, y = random.choice(pv_train); ax[0, j].imshow(Image.open(p)); ax[0, j].set_title("PlantVillage: " + CLASSES[y]); ax[0, j].axis("off")
    src = ext_test if ext_test else pv_test
    p, y = random.choice(src);      ax[1, j].imshow(Image.open(p)); ax[1, j].set_title(("Externo: " if ext_test else "PV: ") + CLASSES[y]); ax[1, j].axis("off")
plt.tight_layout(); plt.show()

## BLOQUE B — Modelos y estrategias de entrenamiento

Tres estrategias para cada arquitectura de `torchvision`:

| Estrategia | Descripción | Learning rate máx. |
|---|---|---|
| `scratch` | Pesos aleatorios, todo se entrena | 0.05 |
| `feature_extraction` | **Transfer learning:** pesos de ImageNet congelados, solo se entrena la capa final | 0.1 (capa final) |
| `finetune` | **Fine-tuning:** pesos de ImageNet, se entrena todo con LR *diferencial* (backbone bajo, cabeza alto) | 0.01 backbone / 0.1 cabeza |

Arquitecturas: ResNet-18/34/50, ResNeXt-50 (32×4d) y Wide ResNet-50-2.

In [ ]:
def build_model(arch, mode, nc=NC):
    weights = None if mode == "scratch" else "DEFAULT"
    m = getattr(models, arch)(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, nc)
    if mode == "feature_extraction":
        for n, p in m.named_parameters():
            p.requires_grad = n.startswith("fc.")
    return m

def make_groups(model, mode):
    head = list(model.fc.parameters()); head_ids = {id(p) for p in head}
    if mode == "scratch":
        return [{"params": list(model.parameters())}], 0.05
    if mode == "feature_extraction":
        return [{"params": head}], 0.1
    back = [p for p in model.parameters() if id(p) not in head_ids]
    return [{"params": back}, {"params": head}], [0.01, 0.1]

def metrics(y, p, prefix):
    present = np.unique(y)
    rec = recall_score(y, p, labels=list(range(NC)), average=None, zero_division=0)
    out = {f"acc_{prefix}": accuracy_score(y, p),
           f"f1_{prefix}": f1_score(y, p, labels=present, average="macro", zero_division=0)}
    for c in range(NC):
        out[f"rec_{CLASSES[c]}_{prefix}"] = rec[c] if c in present else np.nan
    return out

RES_B, KEEP, PREDS, FLOPS = [], {}, {}, {}

def run_experiment(arch, mode, aug, seed, extra_train=None, tag_extra="", keep=False):
    set_seed(seed)
    model = build_model(arch, mode)
    total, trainable = count_params(model)
    if arch not in FLOPS: FLOPS[arch] = count_flops(model, IMG)
    groups, max_lr = make_groups(model, mode)
    train_items = pv_train + (extra_train or [])
    tr = mk(train_items, TF[aug], shuffle=True, drop_last=True)
    t0 = time.time()
    hist, model = fit(model, tr, val_loader, EPOCHS_B, max_lr, groups, wd=1e-4, select_best=True,
                      verbose=False, tag=f"{arch}/{mode}")
    row = dict(arch=arch, mode=mode, aug=aug, extra=tag_extra, seed=seed,
               params_M=total / 1e6, trainable_M=trainable / 1e6, GFLOPs=FLOPS[arch],
               train_time_s=time.time() - t0, best_val=max(h["val_acc"] for h in hist))
    for name, loader in LOADERS_EVAL.items():
        torch.manual_seed(123)                       # mismo ruido del proxy en todas las corridas
        p, y = predict(model, loader)
        row.update(metrics(y, p, name))
        if keep: PREDS[(arch, mode, aug, tag_extra, name)] = (y, p)
    if keep:
        KEEP[(arch, mode, aug, tag_extra)] = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    msg = " | ".join(f"{k}={row[f'acc_{k}']:.3f}" for k in LOADERS_EVAL)
    print(f"{arch:>16} {mode:>18} aug={aug:<6} {tag_extra:<6} seed={seed} → {msg}  ({row['train_time_s']:.0f}s)")
    del model; torch.cuda.empty_cache()
    return row

### B1. Arquitecturas × estrategias (data augmentation débil)

In [ ]:
GRID_B = [(a, m) for a in ["resnet18", "resnet34", "resnet50"] for m in ["scratch", "feature_extraction", "finetune"]]
GRID_B += [("resnext50_32x4d", "finetune"), ("wide_resnet50_2", "finetune")]

for arch, mode in GRID_B:
    for seed in SEEDS:
        RES_B.append(run_experiment(arch, mode, "weak", seed, keep=(seed == SEEDS[0])))

df_B = pd.DataFrame(RES_B); df_B.to_csv(f"{SAVE_DIR}/bloque_B.csv", index=False)

In [ ]:
cols = [c for c in ["acc_intra", "f1_intra", "acc_proxy", "acc_externo", "f1_externo"] if c in df_B.columns]
base = df_B[(df_B.aug == "weak") & (df_B.extra == "")]
tabla_B = base.groupby(["arch", "mode"])[cols + ["params_M", "trainable_M", "GFLOPs", "train_time_s"]].agg(["mean", "std"]).round(4)
tabla_B

### C1. Data augmentation fuerte (mitigación del cambio de dominio)
Se repiten algunas configuraciones con augmentation fuerte (color, rotación, desenfoque, borrado aleatorio) y se compara la **caída** entre test interno y test externo/proxy.

In [ ]:
GRID_C1 = [("resnet18", "finetune"), ("resnet50", "finetune"), ("resnet50", "scratch")]
for arch, mode in GRID_C1:
    for seed in SEEDS:
        RES_B.append(run_experiment(arch, mode, "strong", seed, keep=(seed == SEEDS[0])))
df_B = pd.DataFrame(RES_B); df_B.to_csv(f"{SAVE_DIR}/bloque_B_C.csv", index=False)

### C2. Entrenar mezclando dominios (requiere dataset externo)
Se añade al entrenamiento la mitad del dataset externo (`ext_mix`) y se evalúa en la otra mitad (`ext_test`, que nunca se vio). Es la mitigación más fuerte y a la vez la que más se parece a un escenario real.

In [ ]:
if ext_mix:
    for arch, mode in [("resnet18", "finetune"), ("resnet50", "finetune")]:
        for seed in SEEDS:
            RES_B.append(run_experiment(arch, mode, "strong", seed, extra_train=ext_mix, tag_extra="mixto", keep=(seed == SEEDS[0])))
    df_B = pd.DataFrame(RES_B); df_B.to_csv(f"{SAVE_DIR}/bloque_B_C.csv", index=False)
else:
    print("Sin dataset externo → se omite C2.")

## Resultados: cambio de dominio

In [ ]:
df_B["gap_proxy"] = df_B["acc_intra"] - df_B["acc_proxy"]
if "acc_externo" in df_B: df_B["gap_externo"] = df_B["acc_intra"] - df_B["acc_externo"]

gcols = [c for c in ["acc_intra", "acc_proxy", "acc_externo", "gap_proxy", "gap_externo"] if c in df_B.columns]
resumen = df_B.groupby(["arch", "mode", "aug", "extra"])[gcols].mean().round(4).reset_index()
resumen.to_csv(f"{SAVE_DIR}/resumen_dominio.csv", index=False)
display(resumen)

plot_df = resumen.copy(); plot_df["config"] = plot_df.arch + "\n" + plot_df["mode"] + "\n" + plot_df.aug + plot_df.extra.apply(lambda s: "+" + s if s else "")
m = plot_df.melt(id_vars="config", value_vars=[c for c in ["acc_intra", "acc_proxy", "acc_externo"] if c in plot_df.columns],
                 var_name="evaluación", value_name="accuracy")
plt.figure(figsize=(max(12, 0.9 * len(plot_df)), 5))
sns.barplot(data=m, x="config", y="accuracy", hue="evaluación")
plt.xticks(rotation=90, fontsize=7); plt.ylim(0, 1.02); plt.title("Accuracy: interno vs. proxy vs. externo")
plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/fig_dominio.png", dpi=200); plt.show()

In [ ]:
# Matrices de confusión del mejor modelo (por accuracy interno) en cada dominio
cand = df_B[(df_B.seed == SEEDS[0]) & (df_B.extra == "")].sort_values("acc_intra", ascending=False).iloc[0]
key_best = (cand.arch, cand["mode"], cand.aug, "")
print("Modelo de referencia:", key_best)
fig, ax = plt.subplots(1, len(LOADERS_EVAL), figsize=(5 * len(LOADERS_EVAL), 4.2))
ax = np.atleast_1d(ax)
for a, name in zip(ax, LOADERS_EVAL):
    y, p = PREDS[key_best + (name,)]
    cm = confusion_matrix(y, p, labels=range(NC), normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=a, vmin=0, vmax=1)
    a.set_title(f"{name}"); a.set_xlabel("Predicho"); a.set_ylabel("Real")
plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/fig_confusion.png", dpi=200); plt.show()

## BLOQUE D — Interpretabilidad con Grad-CAM

Se aplica Grad-CAM sobre la última capa convolucional (`layer4[-1]`) para ver **en qué zona se fija la red**. Preguntas clave:
- ¿Mira la lesión de la hoja o el fondo?
- ¿Cambia la atención entre el dominio de laboratorio y el externo/proxy?
- ¿Cómo difieren `scratch`, `finetune` y el entrenamiento con augmentation fuerte?

Se muestran primero las imágenes **mal clasificadas** (si las hay), que son las más informativas.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def load_kept(key):
    m = build_model(key[0], "scratch"); m.load_state_dict(KEEP[key]); return m.to(DEVICE).eval()

def denorm(t):
    mean = torch.tensor(MEAN).view(3, 1, 1); std = torch.tensor(STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

def pick_items(model, items, n, tf=eval_tf, seed=0):
    rng = random.Random(seed); sample = rng.sample(items, min(len(items), 300)); wrong, right = [], []
    for p, y in sample:
        x = tf(Image.open(p).convert("RGB")).unsqueeze(0).to(DEVICE)
        with torch.no_grad(): pred = model(x).argmax(1).item()
        (wrong if pred != y else right).append((p, y))
    return (wrong + right)[:n]

def gradcam_grid(key, items, n=6, tf=eval_tf, title=""):
    model = load_kept(key)
    cam = GradCAM(model=model, target_layers=[model.layer4[-1]])
    chosen = pick_items(model, items, n, tf)
    fig, ax = plt.subplots(2, len(chosen), figsize=(3 * len(chosen), 6.4)); ax = np.array(ax).reshape(2, -1)
    for i, (p, y) in enumerate(chosen):
        torch.manual_seed(i)
        x = tf(Image.open(p).convert("RGB")).unsqueeze(0).to(DEVICE)
        with torch.no_grad(): pred = model(x).argmax(1).item()
        gc = cam(input_tensor=x, targets=[ClassifierOutputTarget(pred)])[0]
        rgb = denorm(x[0].cpu())
        ax[0, i].imshow(rgb); ax[0, i].axis("off"); ax[0, i].set_title(f"Real: {CLASSES[y]}", fontsize=8)
        ax[1, i].imshow(show_cam_on_image(rgb, gc, use_rgb=True)); ax[1, i].axis("off")
        ax[1, i].set_title(f"Pred: {CLASSES[pred]}", fontsize=8, color="green" if pred == y else "red")
    plt.suptitle(title); plt.tight_layout(); plt.show()
    del model, cam; torch.cuda.empty_cache()

key_ft  = ("resnet50", "finetune", "weak", "")
key_sc  = ("resnet50", "scratch",  "weak", "")
key_str = ("resnet50", "finetune", "strong", "")
ext_src = ext_test if ext_test else pv_test
ext_tf  = eval_tf if ext_test else proxy_tf
ext_lbl = "EXTERNO" if ext_test else "PROXY"

for key, nm in [(key_ft, "ResNet-50 fine-tuning (aug. débil)"), (key_sc, "ResNet-50 desde cero"), (key_str, "ResNet-50 fine-tuning (aug. fuerte)")]:
    if key in KEEP:
        gradcam_grid(key, pv_test, tf=eval_tf, title=f"{nm} — dominio PlantVillage (laboratorio)")
        gradcam_grid(key, ext_src, tf=ext_tf, title=f"{nm} — dominio {ext_lbl}")

## BLOQUE E — Costo computacional y despliegue en celular

Para un uso real en zonas con conectividad limitada (como la sierra de Apurímac) importan el **tamaño** y la **velocidad**. Se resume parámetros, FLOPs y se exporta un modelo a **ONNX** (base para llevarlo a móvil con TFLite/ONNX Runtime/ExecuTorch en trabajos futuros).

In [ ]:
costo = (df_B.groupby("arch")[["params_M", "GFLOPs"]].mean().round(2)
         .join(df_B.groupby("arch")["train_time_s"].mean().round(0)))
costo["tamaño_fp32_MB"] = (costo.params_M * 4).round(1)

# Latencia de inferencia (batch=1)
def latencia(arch, n=50):
    m = build_model(arch, "scratch").to(DEVICE).eval(); x = torch.randn(1, 3, IMG, IMG, device=DEVICE)
    with torch.no_grad():
        for _ in range(10): m(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t = time.time()
        for _ in range(n): m(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t) / n * 1000
costo["latencia_ms(GPU)"] = [round(latencia(a), 1) for a in costo.index]
costo.to_csv(f"{SAVE_DIR}/costo_computacional.csv"); display(costo)

# Exportación ONNX del mejor ResNet-18 afinado (si existe)
key_onnx = ("resnet18", "finetune", "strong", "")
if key_onnx not in KEEP: key_onnx = ("resnet18", "finetune", "weak", "")
try:
    m = load_kept(key_onnx).cpu().eval()
    path = f"{SAVE_DIR}/resnet18_papa.onnx"
    torch.onnx.export(m, torch.randn(1, 3, IMG, IMG), path, input_names=["imagen"], output_names=["logits"],
                      opset_version=17, dynamo=False)
    print(f"ONNX exportado: {path} ({os.path.getsize(path)/1e6:.1f} MB)")
except Exception as e:
    print("No se pudo exportar a ONNX:", e)

In [ ]:
# Guardar todo
df_B.to_csv(f"{SAVE_DIR}/todos_los_resultados_B.csv", index=False)
print("Archivos en", SAVE_DIR); print(os.listdir(SAVE_DIR))
# Para descargar desde Colab:  !zip -r resultados.zip /content/resultados  y luego  files.download("resultados.zip")

## Guía para redactar la monografía con estos resultados

1. **Reporta media ± desviación estándar** (3 semillas, `QUICK_MODE=False`). No saques conclusiones con 1 semilla.
2. **Bloque A:** comenta el error de *entrenamiento* (no solo el de prueba): ahí está la evidencia de degradación.
3. **Bloque B:** compara `scratch` vs `feature_extraction` vs `finetune` (¿cuánto aporta ImageNet con ~2 000 imágenes?) y profundidad/variante frente a costo (parámetros, FLOPs, latencia).
4. **Bloque C:** reporta la caída `acc_intra → acc_externo` y si la mitigan el augmentation fuerte o la mezcla de dominios. Si solo usaste el *proxy*, dilo explícitamente como limitación.
5. **Bloque D:** analiza cualitativamente si la atención está en la lesión o en el fondo, especialmente en los errores.
6. **Limitaciones a declarar:** PlantVillage es de laboratorio; el dataset externo puede ser de otra región/variedad (no necesariamente andina); no hay validación en campo en Apurímac; el modelo es solo con fines académicos.
7. **Referencias clave:** He et al. (2016) *Deep Residual Learning*; He et al. (2016) *Identity Mappings*; Xie et al. (2017) *ResNeXt*; Zagoruyko & Komodakis (2016) *Wide ResNet*; Selvaraju et al. (2017) *Grad-CAM*; Hughes & Salathé (2015) *PlantVillage*; Yosinski et al. (2014) *How transferable are features in deep neural networks?*